In [1]:
text = open("../data/splits/train.txt", encoding="utf-8").read()
chars = sorted(set(text))
print(len(chars))
print("".join(chars))

352

 !"#$%&'()*+,-./0123456789:;<=>?@ABCDEFGHIJKLMNOPQRSTUVWXYZ[\]^_abcdefghijklmnopqrstuvwxyz{|}~§¨©¬®²³¶·¹¼½ÁÃÈÉÊÍÏÑÕÚÜàáâãäæçèéêëíïðñòóôõöøúüýĀāăčĐēėĨĩĭİıĶľŀłŅŋŌōŏœśŞšţŨũŬŭůŰűźŻưǐǯȁȅȉșțȳɑɔəɛɨɪʉʹʽʾʿ̷̨̃̄̇αβεθАаекоуіՏׁהוינפשءآأؤإئابةتثجحخدذرزسشصضطظعغفقكلمنهوىيًٌٍَُِّْٰนႮღᵁᵉᵏḁḍḡḤḥḵḷṁṃṅṇṉṛṠṡṢṣṫṬṭṯṱṵṹṼṽẏạảẩắẽềệọồổờỡứữỹἐῖῦ‐–—‟•…›⁄€™→∙⑥⑨□▪▲▸○●☺♀♬✭❤「踽💎😀😂😅😊🙄🤗🤬


In [2]:
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

In [3]:
print(stoi["k"])
print(itos[stoi["k"]])

75
k


In [4]:
import torch

data = torch.tensor([stoi[c] for c in text])
print(len(data))
print(data[:20])

9506558
tensor([168,  84,  79,  78,  89,  65,   1,  75,  87, 150,  75,  65,   1,  65,
         84,  65,   1, 150,  76,  65])


In [5]:
V = len(chars)
ids = data.tolist()
N = [[0] * V for _ in range(V)]
for a, b in zip(ids, ids[1:]):
    N[a][b] += 1
N = torch.tensor(N)


In [6]:
print(N[stoi["k"], stoi["a"]])

tensor(106072)


In [7]:
P = (N + 1).float()
P = P / P.sum(dim=1, keepdim=True)
print(P[stoi["k"]].sum())

tensor(1.0000)


In [8]:
g = torch.Generator().manual_seed(42)
cur = stoi[" "]
out = []
for _ in range(300):
    cur = torch.multinomial(P[cur], 1, generator=g).item()
    out.append(itos[cur])
print("".join(out))

ka Nĩandĩ nĩtwatu, kanya Kwako 'osyai mbingi ma a kanaloklike seangokĩĩ HLAAma yaininĩteenandũ inĩi Ũnitũnya o, 20. Na Iali ĩwo Nĩya anĩĩka touki ndena vi ũ twovi nano nĩ Mbina a nthĩ wonaa ngúZæŭmwa?" ya nta mwa ndũlani, Kyatũkw'ayanya kwona kwa Tũilai, ia i Ngĩ kw'manani mu kanantenyaĩta sekwēč³بי


In [9]:
import math

val_text = open("../data/splits/val.txt", encoding="utf-8").read()
total, n = 0.0, 0
for a, b in zip(val_text, val_text[1:]):
    if a in stoi and b in stoi:
        total += -math.log2(P[stoi[a], stoi[b]].item())
        n += 1
print(total / n)

3.298018390779561


In [10]:
import json
json.dump({"model": "bigram (counting, +1 smoothing)", "val_bpc": total / n},
          open("../results/bigram.json", "w"), indent=2)

In [14]:
import torch.nn as nn

class BigramLM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx):
        return self.table(idx)

m = BigramLM(V)
x = torch.tensor([[1, 2, 3]])
print(m(x).shape)

torch.Size([1, 3, 352])


In [15]:
block = 8
s = torch.randint(0, len(data) - block - 1, (1,)).item()
x = data[s : s + block]
y = data[s + 1 : s + block + 1]
print("".join(itos[i.item()] for i in x))
print("".join(itos[i.item()] for i in y))

 kwa Esa
kwa Esaũ


In [21]:
def get_batch(data, block, batch_size):
    s = torch.randint(0, len(data) - block - 1, (batch_size,))
    x = [data[i : i + block] for i in s]
    y = [data[i + 1: i + block + 1] for i in s]
    x = torch.stack(x)
    y = torch.stack(y)
    return x, y

x, y = get_batch(data, 8, 4)
print(x.shape, y.shape)

torch.Size([4, 8]) torch.Size([4, 8])


In [20]:
import torch.nn.functional as F

m = BigramLM(V)
x, y = get_batch(data, 8, 4)
logits = m(x)
loss = F.cross_entropy(logits.view(-1, V), y.view(-1))
print(loss.item())

tensor([5899311, 1154190, 2276689, 8226801])
6.630352020263672


In [23]:
optimizer = torch.optim.AdamW(m.parameters(), lr=0.01)

for step in range(2000):
    x, y = get_batch(data, 8, 32)
    logits = m(x)
    loss = F.cross_entropy(logits.view(-1, V), y.view(-1))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step % 200 == 0:
        print(step, loss.item())

0 2.2432332038879395
200 2.34291934967041
400 2.4566073417663574
600 2.2277843952178955
800 2.3179397583007812
1000 2.3084473609924316
1200 2.2132744789123535
1400 2.258997678756714
1600 2.156771421432495
1800 2.304396152496338


In [25]:
text_plain = text.replace("ĩ", "i").replace("ũ", "u").replace("Ĩ", "I").replace("Ũ", "U")
print(text_plain[:200])

Utonya kwika ata ila angi makwika undu utaile?

Muno muno liu uu niwaetae uvosyo kwa ala awau na syana.

Andu aa mekua anene ma miao Silikalini na ala ma muvia wa ita wa kuthiana maundu ta matesi wia 


In [26]:
chars_plain = sorted(set(text_plain))
print(len(chars_plain))

348


In [28]:
stoi_plain = {ch: i for i, ch in enumerate(chars_plain)}
itos_plain = {i: ch for i, ch in enumerate(chars_plain)}
print(stoi_plain["u"], itos_plain[stoi_plain["u"]])

85 u


In [29]:
V_plain = len(chars_plain)
ids_plain = [stoi_plain[c] for c in text_plain]
N_plain = [[0] * V_plain for _ in range(V_plain)]
for a, b in zip(ids_plain, ids_plain[1:]):
    N_plain[a][b] += 1
N_plain = torch.tensor(N_plain)
print(N_plain[stoi_plain["k"], stoi_plain["a"]])

tensor(106072)


In [31]:
P_plain = (N_plain + 1).float()
P_plain = P_plain / P_plain.sum(dim=1, keepdim=True)
print(P_plain[stoi_plain["k"]].sum())

tensor(1.0000)


In [34]:
def sample(P, stoi, itos, seed):
    g = torch.Generator().manual_seed(seed)
    cur = stoi[" "]
    out = []
    for _ in range(100):
        cur = torch.multinomial(P[cur], 1, generator=g).item()
        out.append(itos[cur])
    return "".join(out)

print(sample(P, stoi, itos, 1))
print(sample(P_plain, stoi_plain, itos_plain, 1))

ũsyanoona nyana "Aviyũangeto ũva nĩ a mamise? a 7:1215.
1414:1-mũnyandunisunyaũu ũtĩva kĩsithae kũnd
mavani kungale itheonziwa vyoo Ma nga-a tw'e ilu ma unya u la kuomaekwe a wae ibithwa.—Aiatuinthili 


In [35]:
pairs = []
for s in range(1, 6):
    pairs.append((sample(P, stoi, itos, s), "tildes"))
    pairs.append((sample(P_plain, stoi_plain, itos_plain, s), "plain"))
print(len(pairs))

10


In [39]:
import random
random.Random(7).shuffle(pairs)
for i, (text, label) in enumerate(pairs, 1):

    print(f"=== Sentence {i} ===")
    print(text.replace("\n", " ").replace("ĩ", "i").replace("ũ", "u").replace("Ĩ", "I").replace("Ũ", "U"))
    print()

=== Sentence 1 ===
mavani kungale itheonziwa vyoo Ma nga-a tw'e ilu ma unya u la kuomaekwe a wae ibithwa.—Aiatuinthili 

=== Sentence 2 ===
syo iw'o 4, kalaiaiaakaka nanilunivonivi yukatha umutyaonyiakiu kla vilumathilivilesyo 4) KWellusewe

=== Sentence 3 ===
ngungo mo ku nili? 5.'itu ndesthe ta mi, ka (27. 2: Sa kuna. Pi su nyevinailiona mbekikama ka kwa ny

=== Sentence 4 ===
suleo nyaunduniana yakalamwetainee muunza naa thuinikundu ndu kua ioonga aki u enithya e. Kusyinyaat

=== Sentence 5 ===
nze thwo ndutwauyi ulii wawonyasumuvoonisona mu kuthilaka ivuyanga iteme, maaki nduteeu NG‐يَṢٰ*ổؤآṵ

=== Sentence 6 ===
tho e;ZIKu Asusyeki, mu si wandilu ni ma nong'  Sa, ete munga a arindi mwa ma landanyai ma ivetasu v

=== Sentence 7 ===
usyanoona nyana "Aviyuangeto uva ni a mamise? a 7:1215. 1414:1-munyandunisunyauu utiva kisithae kund

=== Sentence 8 ===
nde. Inite u. nani ku tana Avu tatume maki avavenauna Mwanek.  MBetwa, akama kunye"uuna VO meva onia

=== Sentence 9 ===
thwe Osa uma 

In [40]:
with open("../results/blind_tilde_test.txt", "w", encoding="utf-8") as f:
    for i, (text, label) in enumerate(pairs, 1):
        f.write(f"{i} | {label} | {text}\n\n")